In [6]:
import ffmpeg
import whisper
from gtts import gTTS
from deep_translator import GoogleTranslator
import os
os.environ["PATH"] += os.pathsep + r"C:\ffmpeg\bin"

# === Arquivos ===
input_file = "Wellington.mp4"             # vídeo original
extracted_audio = "temp_audio.wav"        # áudio extraído do vídeo
translated_audio = "final_pt_audio.mp3"   # novo áudio em português
output_file = "Wellington_traduzido.mp4"  # vídeo final traduzido

# === 1. Extrair áudio original ===
ffmpeg.input(input_file).output(extracted_audio).run(cmd="C:/ffmpeg/bin/ffmpeg.exe", overwrite_output=True)


# === 2. Transcrever em inglês ===
model = whisper.load_model("small")
#result = model.transcribe(r"C:\Users\calva\OneDrive\Documentos\CodesOptimizationReservoirModels\codes_Proxy\temp_audio.wav", language="en")
result = model.transcribe(extracted_audio, language="en")


# === 3. Traduzir texto em blocos (máx. 4500 caracteres cada) ===
def translate_large_text(text, source="en", target="pt", max_chars=4500):
    translator = GoogleTranslator(source=source, target=target)
    translated_parts = []
    for i in range(0, len(text), max_chars):
        chunk = text[i:i+max_chars]
        translated = translator.translate(chunk)
        translated_parts.append(translated)
    return " ".join(translated_parts)

full_text_en = result["text"]
full_text_pt = translate_large_text(full_text_en)

# Salvar tradução em arquivo de texto (útil para revisão)
with open("traducao_portugues.txt", "w", encoding="utf-8") as f:
    f.write(full_text_pt)

# === 4. Gerar áudio em português ===
tts = gTTS(full_text_pt, lang="pt")
tts.save(translated_audio)

# === 5. Substituir áudio original pelo traduzido ===
in_video = ffmpeg.input(input_file)          # vídeo original
in_audio = ffmpeg.input(translated_audio)    # novo áudio traduzido

(
    ffmpeg
    .output(in_video.video, in_audio.audio, output_file, vcodec='copy', acodec='aac')
    .run(overwrite_output=True)
)

print("✅ Vídeo final gerado:", output_file)


✅ Vídeo final gerado: Wellington_traduzido.mp4
